# Notebook 8: Machine Learning Modelling

**Research Project:** Improving Asymmetric Exchange Rate Pass-Through Modelling Across Food Price Categories in South Africa Using Machine Learning

**Modelling Period:** October 2017 – December 2025

## Notebook Objective

This notebook develops machine learning models for predicting monthly food inflation across South African food subclasses.

The analysis will:

1. load the modelling dataset prepared in Notebook 5
2. preserve the predefined chronological data splits
3. compare symmetric and asymmetric exchange-rate feature representations
4. establish persistence and regularised linear benchmarks
5. train Random Forest and XGBoost models
6. select model settings using the validation period only
7. prevent the test period from influencing model development
8. export fitted models and predictions for final evaluation in Notebook 9

The target variable is monthly food inflation. The symmetric models use lagged overall exchange-rate changes, while the asymmetric models use separate lagged depreciation and appreciation features.

Notebook 8 performs model development and validation. Final test evaluation and comparison with the econometric benchmarks are reserved for Notebook 9.

## Machine Learning Strategy

A pooled modelling approach is used across the 46 food subclasses. Each
observation represents one food subclass in one month.

Food-subclass indicators allow the models to account for persistent
category-level differences, while lagged food-inflation and exchange-rate features capture temporal relationships.

Three model families will be considered:

- Ridge regression as a regularised linear benchmark
- Random Forest for nonlinear relationships and interactions
- XGBoost for gradient-boosted tree modelling

A persistence forecast, which uses the previous month's food inflation as the prediction, provides a simple time-series benchmark.

Each model family will be estimated using two feature representations:

- a symmetric representation using lagged total exchange-rate changes
- an asymmetric representation using separate depreciation and appreciation magnitudes.

Models will be trained on observations ending in December 2023. Hyperparameter selection will use the 2024 validation period. The 2025 test period will remain untouched during model development.

In [1]:
# import Libraries

from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import xgboost

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from xgboost import XGBRegressor

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

RANDOM_STATE = 42

print("Libraries imported successfully.")
print("scikit-learn version:", sklearn.__version__)
print("XGBoost version:", xgboost.__version__)

Libraries imported successfully.
scikit-learn version: 1.9.0
XGBoost version: 3.3.0


In [2]:
# load and validate the data handoff

ml_data_path = Path(
    "../data/processed/ml_model_data.csv"
)

if not ml_data_path.exists():
    raise FileNotFoundError(
        f"Machine learning dataset not found: {ml_data_path}"
    )

ml_data = pd.read_csv(
    ml_data_path,
    parse_dates=["Date"],
)

ml_data = (
    ml_data
    .sort_values(["Date", "SubclassDescription"])
    .reset_index(drop=True)
)

required_handoff_columns = {
    "Date",
    "ClassDescription",
    "SubclassDescription",
    "Subclass_Weight",
    "Food_Inflation_Pct",
    "Split",
}

missing_handoff_columns = (
    required_handoff_columns.difference(ml_data.columns)
)

if missing_handoff_columns:
    raise ValueError(
        "Missing handoff columns: "
        f"{sorted(missing_handoff_columns)}"
    )

handoff_summary = pd.DataFrame(
    {
        "Value": [
            len(ml_data),
            ml_data["SubclassDescription"].nunique(),
            ml_data["Date"].nunique(),
            ml_data["Date"].min(),
            ml_data["Date"].max(),
            ml_data.isna().sum().sum(),
            ml_data.duplicated(
                ["Date", "SubclassDescription"]
            ).sum(),
        ]
    },
    index=[
        "Observations",
        "Food subclasses",
        "Unique months",
        "Start date",
        "End date",
        "Missing values",
        "Duplicate subclass-month rows",
    ],
)

display(handoff_summary)

,Value
Observations,4554
Food subclasses,46
Unique months,99
Start date,2017-10-01 00:00:00
End date,2025-12-01 00:00:00
Missing values,0
Duplicate subclass-month rows,0


In [3]:
# define target and features

target_column = "Food_Inflation_Pct"

categorical_features = [
    "SubclassDescription",
]

shared_numeric_features = [
    "Food_Inflation_Lag1_Pct",
    "Food_Inflation_Lag3_Pct",
    "Food_Inflation_Lag6_Pct",
    "Month_Sin",
    "Month_Cos",
]

symmetric_exchange_features = [
    "ExchangeRate_Change_Lag1_Pct",
    "ExchangeRate_Change_Lag3_Pct",
    "ExchangeRate_Change_Lag6_Pct",
]

asymmetric_exchange_features = [
    "Depreciation_Shock_Lag1_Pct",
    "Depreciation_Shock_Lag3_Pct",
    "Depreciation_Shock_Lag6_Pct",
    "Appreciation_Magnitude_Lag1_Pct",
    "Appreciation_Magnitude_Lag3_Pct",
    "Appreciation_Magnitude_Lag6_Pct",
]

symmetric_model_features = (
    categorical_features
    + shared_numeric_features
    + symmetric_exchange_features
)

asymmetric_model_features = (
    categorical_features
    + shared_numeric_features
    + asymmetric_exchange_features
)

required_model_columns = set(
    symmetric_model_features
    + asymmetric_model_features
    + [target_column]
)

missing_model_columns = required_model_columns.difference(
    ml_data.columns
)

if missing_model_columns:
    raise ValueError(
        "Missing modelling columns: "
        f"{sorted(missing_model_columns)}"
    )

feature_summary = pd.DataFrame(
    {
        "Representation": [
            "Symmetric",
            "Asymmetric",
        ],
        "Categorical_Features": [
            len(categorical_features),
            len(categorical_features),
        ],
        "Numeric_Features": [
            (
                len(shared_numeric_features)
                + len(symmetric_exchange_features)
            ),
            (
                len(shared_numeric_features)
                + len(asymmetric_exchange_features)
            ),
        ],
        "Total_Input_Columns": [
            len(symmetric_model_features),
            len(asymmetric_model_features),
        ],
    }
)

display(feature_summary)

,Representation,Categorical_Features,Numeric_Features,Total_Input_Columns
0,Symmetric,1,8,9
1,Asymmetric,1,11,12


In [4]:
# create chronological data splits

split_order = ["Train", "Validation", "Test"]

unexpected_splits = set(
    ml_data["Split"].unique()
).difference(split_order)

if unexpected_splits:
    raise ValueError(
        f"Unexpected split labels: {sorted(unexpected_splits)}"
    )

train_data = ml_data.loc[
    ml_data["Split"] == "Train"
].copy()

validation_data = ml_data.loc[
    ml_data["Split"] == "Validation"
].copy()

test_data = ml_data.loc[
    ml_data["Split"] == "Test"
].copy()

if not (
    train_data["Date"].max()
    < validation_data["Date"].min()
    <= validation_data["Date"].max()
    < test_data["Date"].min()
):
    raise ValueError(
        "The chronological split boundaries are invalid."
    )

split_summary = (
    ml_data
    .groupby("Split")
    .agg(
        Start_Date=("Date", "min"),
        End_Date=("Date", "max"),
        Observations=("Date", "size"),
        Food_Subclasses=(
            "SubclassDescription",
            "nunique",
        ),
        Unique_Months=("Date", "nunique"),
    )
    .reindex(split_order)
)

display(split_summary)

print(
    "Training ends before validation:",
    train_data["Date"].max()
    < validation_data["Date"].min(),
)

print(
    "Validation ends before testing:",
    validation_data["Date"].max()
    < test_data["Date"].min(),
)

,Start_Date,End_Date,Observations,Food_Subclasses,Unique_Months
Split,,,,,
Train,2017-10-01,2023-12-01,3450,46,75
Validation,2024-01-01,2024-12-01,552,46,12
Test,2025-01-01,2025-12-01,552,46,12


Training ends before validation: True
Validation ends before testing: True


### Modelling Data Interpretation

The modelling dataset contains 99 monthly observations for each of the 46 food subclasses.

The training period provides 75 months per subclass, while the validation and test periods each contain 12 months per subclass. Every food subclass is represented in all three periods.

The symmetric representation contains eight numeric predictors and one
categorical predictor. The asymmetric representation contains three additional numeric predictors because depreciation and appreciation are represented separately.

The larger asymmetric feature set will be compared with the symmetric set within each model family. This ensures that any improvement is attributed to the exchange-rate representation rather than to a different algorithm or evaluation period.

In [5]:
# create training and validation matrices

model_feature_sets = {
    "Symmetric": symmetric_model_features,
    "Asymmetric": asymmetric_model_features,
}

numeric_feature_sets = {
    "Symmetric": (
        shared_numeric_features
        + symmetric_exchange_features
    ),
    "Asymmetric": (
        shared_numeric_features
        + asymmetric_exchange_features
    ),
}

training_features = {
    representation: train_data[features].copy()
    for representation, features in model_feature_sets.items()
}

validation_features = {
    representation: validation_data[features].copy()
    for representation, features in model_feature_sets.items()
}

training_target = train_data[target_column].copy()
validation_target = validation_data[target_column].copy()

training_subclasses = set(
    train_data["SubclassDescription"]
)
validation_subclasses = set(
    validation_data["SubclassDescription"]
)
test_subclasses = set(
    test_data["SubclassDescription"]
)

print(
    "Training target observations:",
    len(training_target),
)
print(
    "Validation target observations:",
    len(validation_target),
)
print(
    "Validation subclasses unseen during training:",
    len(validation_subclasses - training_subclasses),
)
print(
    "Test subclasses unseen during training:",
    len(test_subclasses - training_subclasses),
)

Training target observations: 3450
Validation target observations: 552
Validation subclasses unseen during training: 0
Test subclasses unseen during training: 0


In [6]:
# define preprocessing pipelines

def create_preprocessor(
    numeric_features,
    scale_numeric,
):
    numeric_transformer = (
        StandardScaler()
        if scale_numeric
        else "passthrough"
    )

    return ColumnTransformer(
        transformers=[
            (
                "category",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                categorical_features,
            ),
            (
                "numeric",
                numeric_transformer,
                numeric_features,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


ridge_preprocessors = {
    representation: create_preprocessor(
        numeric_features=numeric_features,
        scale_numeric=True,
    )
    for representation, numeric_features
    in numeric_feature_sets.items()
}

tree_preprocessors = {
    representation: create_preprocessor(
        numeric_features=numeric_features,
        scale_numeric=False,
    )
    for representation, numeric_features
    in numeric_feature_sets.items()
}

preprocessing_records = []

for representation in model_feature_sets:
    fitted_preprocessor = clone(
        ridge_preprocessors[representation]
    )

    transformed_training_data = (
        fitted_preprocessor.fit_transform(
            training_features[representation]
        )
    )

    preprocessing_records.append(
        {
            "Representation": representation,
            "Input_Columns": len(
                model_feature_sets[representation]
            ),
            "Transformed_Features": (
                transformed_training_data.shape[1]
            ),
            "Training_Observations": (
                transformed_training_data.shape[0]
            ),
        }
    )

preprocessing_summary = pd.DataFrame(
    preprocessing_records
)

display(preprocessing_summary)

,Representation,Input_Columns,Transformed_Features,Training_Observations
0,Symmetric,9,54,3450
1,Asymmetric,12,57,3450


In [7]:
# evaluate the validation benchmark

def calculate_regression_metrics(
    actual_values,
    predicted_values,
):
    actual = np.asarray(actual_values, dtype=float)
    predicted = np.asarray(
        predicted_values,
        dtype=float,
    )

    return {
        "MAE": mean_absolute_error(
            actual,
            predicted,
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                actual,
                predicted,
            )
        ),
        "R2": r2_score(
            actual,
            predicted,
        ),
        "Directional_Accuracy_Pct": (
            np.mean(
                np.sign(actual)
                == np.sign(predicted)
            )
            * 100
        ),
    }


persistence_validation_predictions = validation_data[
    "Food_Inflation_Lag1_Pct"
].to_numpy()

persistence_metrics = calculate_regression_metrics(
    validation_target,
    persistence_validation_predictions,
)

validation_model_results = pd.DataFrame(
    [
        {
            "Model": "Persistence",
            "Representation": "Lag-1 benchmark",
            **persistence_metrics,
        }
    ]
)

display(validation_model_results)

,Model,Representation,MAE,RMSE,R2,Directional_Accuracy_Pct
0,Persistence,Lag-1 benchmark,1.352852,2.340242,-0.519088,56.340580
